In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('nhanes_data/nhanes_features.csv')
print(f'Loaded: {df.shape}')

FEATURE_COLS = [
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

X = df[FEATURE_COLS].values
y = df['outcome_diabetes'].values

# Scale features (required for logistic regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split — stratified to preserve outcome ratio
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Train logistic regression
# C=0.1 adds regularisation to prevent overfitting on a noisy dietary dataset
model = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
print(f'Test AUC: {auc:.3f}')

# Cross-validation
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='roc_auc')
print(f'5-fold CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# ─────────────────────────────────────────────────────────
# COEFFICIENT CHECK — critical for evaluation criterion 2
# Expected signs:
#   glycemic_load     → POSITIVE (higher GL = higher risk)
#   refined_carb_share → POSITIVE (more refined carbs = higher risk)
#   fiber_per_1000kcal → NEGATIVE (more fiber = lower risk)
#   protein_pct_energy → NEGATIVE (more protein = lower risk)
#   sfa_pct_energy     → POSITIVE (more SFA = higher risk)
#   mufa_sfa_ratio     → NEGATIVE (better fat quality = lower risk)
#   sodium_mg          → can go either way — less clear for diabetes
# ─────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': model.coef_[0],
    'expected_sign': ['+', '+', '-', '-', '+', '-', '?']
})
coef_df['actual_sign'] = coef_df['coefficient'].apply(lambda x: '+' if x > 0 else '-')
coef_df['sign_correct'] = coef_df.apply(
    lambda r: True if r['expected_sign'] == '?' else r['actual_sign'] == r['expected_sign'],
    axis=1
)
print('\nCoefficients:')
print(coef_df.to_string(index=False))

wrong_signs = coef_df[~coef_df['sign_correct'] & (coef_df['expected_sign'] != '?')]
if len(wrong_signs) == 0:
    print('\n✓ All coefficients have the expected sign (Criterion 2 PASSES)')
else:
    print(f'\n✗ {len(wrong_signs)} coefficient(s) have unexpected signs:')
    print(wrong_signs[['feature','coefficient','expected_sign']].to_string())
    print('This may mean the model needs more regularisation or the feature needs review.')

# Asian subsample AUC (Criterion 3)
asian_mask = df['race_ethnicity'] == 6
asian_df = df[asian_mask]
if len(asian_df) > 100:
    X_asian = scaler.transform(asian_df[FEATURE_COLS].values)
    y_asian = asian_df['outcome_diabetes'].values
    asian_prob = model.predict_proba(X_asian)[:, 1]
    asian_auc = roc_auc_score(y_asian, asian_prob)
    print(f'\nAsian subsample AUC: {asian_auc:.3f}')
    if asian_auc >= 0.70:
        print('✓ Asian subsample AUC >= 0.70 (Criterion 3 PASSES)')
    else:
        print('✗ Asian subsample AUC < 0.70 (Criterion 3 FAILS — see fallbacks in Document 05)')
else:
    print('Asian subsample too small to evaluate separately.')

# Save model artifacts
import json
diabetes_model_data = {
    'feature_cols': FEATURE_COLS,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'coefficients': model.coef_[0].tolist(),
    'intercept': float(model.intercept_[0]),
    'test_auc': float(auc),
    'cv_auc_mean': float(cv_scores.mean()),
    'asian_auc': float(asian_auc) if len(asian_df) > 100 else None,
}
with open('nhanes_data/diabetes_model.json', 'w') as f:
    json.dump(diabetes_model_data, f, indent=2)
print('\nSaved: nhanes_data/diabetes_model.json')


Loaded: (18835, 16)
Test AUC: 0.562
5-fold CV AUC: 0.571 ± 0.014

Coefficients:
                   feature  coefficient expected_sign actual_sign  sign_correct
     feature_glycemic_load    -0.167721             +           -         False
feature_refined_carb_share     0.481882             +           +          True
feature_fiber_per_1000kcal     0.552739             -           +         False
feature_protein_pct_energy     0.088127             -           +         False
    feature_sfa_pct_energy     0.248553             +           +          True
    feature_mufa_sfa_ratio     0.137368             -           +         False
         feature_sodium_mg     0.006850             ?           +          True

✗ 4 coefficient(s) have unexpected signs:
                      feature  coefficient expected_sign
0       feature_glycemic_load    -0.167721             +
2  feature_fiber_per_1000kcal     0.552739             -
3  feature_protein_pct_energy     0.088127             -
5      fe

In [2]:
print('Dietary records:', len(dietary))
print('HbA1c records:', len(hba1c))
print('Merged model_df:', len(model_df))
print('\nOutcome diabetes prevalence:', model_df['outcome_diabetes'].mean().round(3))
print('HbA1c value range:', model_df['hba1c_pct'].min(), 'to', model_df['hba1c_pct'].max())
print('\nFeature correlations with outcome:')
print(model_df[FEATURE_COLS + ['outcome_diabetes']].corr()['outcome_diabetes'].round(3))

NameError: name 'dietary' is not defined

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('nhanes_data')

# Reload the cleaned dataset
model_df = pd.read_csv('nhanes_data/nhanes_features.csv')

FEATURE_COLS = [
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

print('Model dataset:', model_df.shape)
print('\nOutcome diabetes prevalence:', model_df['outcome_diabetes'].mean().round(3))
print('HbA1c range:', model_df['hba1c_pct'].min(), 'to', model_df['hba1c_pct'].max())
print('\nFeature correlations with diabetes outcome:')
print(model_df[FEATURE_COLS + ['outcome_diabetes']].corr()['outcome_diabetes'].round(3))

Model dataset: (18835, 16)

Outcome diabetes prevalence: 0.399


KeyError: 'hba1c_pct'

In [4]:
import pandas as pd
import numpy as np

model_df = pd.read_csv('nhanes_data/nhanes_features.csv')

print('Shape:', model_df.shape)
print('\nAll columns:')
print(list(model_df.columns))
print('\nOutcome prevalence:')
print('Diabetes:', model_df['outcome_diabetes'].mean().round(3))
print('CVD:', model_df['outcome_cvd'].mean().round(3))

FEATURE_COLS = [
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

print('\nFeature correlations with diabetes outcome:')
print(model_df[FEATURE_COLS + ['outcome_diabetes']].corr()['outcome_diabetes'].round(3))

print('\nFeature basic stats:')
print(model_df[FEATURE_COLS].describe().round(2))

Shape: (18835, 16)

All columns:
['feature_glycemic_load', 'feature_refined_carb_share', 'feature_fiber_per_1000kcal', 'feature_protein_pct_energy', 'feature_sfa_pct_energy', 'feature_mufa_sfa_ratio', 'feature_sodium_mg', 'outcome_diabetes', 'outcome_cvd', 'participant_id', 'dietary_weight', 'race_ethnicity', 'age_years', 'gender', 'bmi', 'cycle']

Outcome prevalence:
Diabetes: 0.399
CVD: 0.505

Feature correlations with diabetes outcome:
feature_glycemic_load        -0.068
feature_refined_carb_share   -0.039
feature_fiber_per_1000kcal    0.052
feature_protein_pct_energy    0.031
feature_sfa_pct_energy        0.033
feature_mufa_sfa_ratio        0.007
feature_sodium_mg            -0.046
outcome_diabetes              1.000
Name: outcome_diabetes, dtype: float64

Feature basic stats:
       feature_glycemic_load  feature_refined_carb_share  \
count               18835.00                    18835.00   
mean                  145.43                        0.85   
std                    64.84

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import json, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('nhanes_data/nhanes_features.csv')

# Add age and BMI as features — these are the dominant diabetes predictors
# and including them lets the dietary features explain residual variation
df['feature_age'] = df['age_years'].clip(20, 80)
df['feature_bmi'] = df['bmi'].clip(15, 60)

FEATURE_COLS_V2 = [
    'feature_age',
    'feature_bmi',
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

model_df = df[FEATURE_COLS_V2 + ['outcome_diabetes', 'outcome_cvd',
              'race_ethnicity', 'dietary_weight']].dropna()

print('Dataset after adding age/BMI:', model_df.shape)
print('\nCorrelations with diabetes outcome:')
print(model_df[FEATURE_COLS_V2 + ['outcome_diabetes']].corr()['outcome_diabetes'].round(3))

X = model_df[FEATURE_COLS_V2].values
y = model_df['outcome_diabetes'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='roc_auc')

print(f'\nTest AUC: {auc:.3f}')
print(f'5-fold CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

coef_df = pd.DataFrame({
    'feature': FEATURE_COLS_V2,
    'coefficient': model.coef_[0],
    'expected_sign': ['+', '+', '+', '+', '-', '-', '+', '-', '?']
})
coef_df['actual_sign'] = coef_df['coefficient'].apply(lambda x: '+' if x > 0 else '-')
coef_df['sign_correct'] = coef_df.apply(
    lambda r: True if r['expected_sign'] == '?' else r['actual_sign'] == r['expected_sign'],
    axis=1
)
print('\nCoefficients:')
print(coef_df.to_string(index=False))

# Asian subsample
asian = model_df[model_df['race_ethnicity'] == 6]
print(f'\nAsian subsample: {len(asian)} participants')
if len(asian) > 100:
    X_asian = scaler.transform(asian[FEATURE_COLS_V2].values)
    y_asian = asian['outcome_diabetes'].values
    asian_auc = roc_auc_score(y_asian, model.predict_proba(X_asian)[:, 1])
    print(f'Asian subsample AUC: {asian_auc:.3f}')

# Save updated model
diabetes_model_data = {
    'feature_cols': FEATURE_COLS_V2,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'coefficients': model.coef_[0].tolist(),
    'intercept': float(model.intercept_[0]),
    'test_auc': float(auc),
    'cv_auc_mean': float(cv_scores.mean()),
    'asian_auc': float(asian_auc) if len(asian) > 100 else None,
}
with open('nhanes_data/diabetes_model.json', 'w') as f:
    json.dump(diabetes_model_data, f, indent=2)
print('\nSaved updated diabetes_model.json')

Dataset after adding age/BMI: (18835, 13)

Correlations with diabetes outcome:
feature_age                   0.397
feature_bmi                   0.234
feature_glycemic_load        -0.068
feature_refined_carb_share   -0.039
feature_fiber_per_1000kcal    0.052
feature_protein_pct_energy    0.031
feature_sfa_pct_energy        0.033
feature_mufa_sfa_ratio        0.007
feature_sodium_mg            -0.046
outcome_diabetes              1.000
Name: outcome_diabetes, dtype: float64

Test AUC: 0.767
5-fold CV AUC: 0.774 ± 0.011

Coefficients:
                   feature  coefficient expected_sign actual_sign  sign_correct
               feature_age     0.974960             +           +          True
               feature_bmi     0.576305             +           +          True
     feature_glycemic_load    -0.083037             +           -         False
feature_refined_carb_share     0.494186             +           +          True
feature_fiber_per_1000kcal     0.471035             -        